# Linear Models

This notebook covers `LinearBlock` and `MLP`: the fully-connected primitives in `ml_suite`. `LinearBlock` is a single conditioned projection step; `MLP` stacks N of them into a deep network. Both broadcast over arbitrary leading dimensions, so they drop in as per-token heads, per-frame processors, or standalone regressors.

## Class hierarchy

```
LinearBlock   <- primitive
    └── MLP   <- stacks N LinearBlocks
```

`LinearBlock` is also used internally by:
- `MLP`: every layer in an MLP is a `LinearBlock`
- `ContinuousInputTokenizer` / `TokenwiseHead` / `PooledHead` (transformer module): their projection MLPs are built from the same `make_head_mlp` helper, which assembles `nn.Linear` + activations in the same style

## How to choose

Reach for `LinearBlock` when you need exactly one conditioned projection. Reach for `MLP` whenever you need depth — it handles dimension bookkeeping and wires context to every layer automatically. The conditioning modes (`film`, `concat`, `add`, `multiply`, `cross_attn`) are identical between the two; `MLP` just applies whichever you pick at every layer.

## Task selection guide

| Task | Recommended class |
|---|---|
| Single projection, no conditioning | `LinearBlock` (no context) |
| Single projection with conditioning signal | `LinearBlock` with `film` or `concat` |
| Deep MLP, no conditioning | `MLP` |
| Score network / noise predictor (diffusion) | `MLP` with `film` context injection |
| Learned conditioning adapter (e.g. map CLIP → task space) | `MLP` with `concat` or `film` |
| Per-token processing head on top of a transformer | `MLP` (broadcasts over token dim) |
| Residual refinement block | `LinearBlock(do_residual=True)` or `MLP(do_residual=True)` |

In [15]:
import torch
from ml_suite.models.linear import LinearBlock, MLP

---
## 1. `LinearBlock` : the primitive

A single linear transformation with optional context conditioning and residual.

Operation order: **Context Conditioning -> Linear Projection -> Residual Add -> Activation**

**Integrates into:** `MLP` : every layer in an `MLP` is a `LinearBlock`. Use `LinearBlock` directly when you need exactly one conditioned projection step (e.g. a projection head, a conditioning adapter, or a single-layer score network).

**How a user is meant to use it:** Reach for `LinearBlock` when you need one layer with a specific conditioning mode. If you need depth (more than one layer), use `MLP` instead : it handles dimension bookkeeping and wires context to every layer for you.

### 1.1 Basic — no conditioning

Projects from one feature dimension to another with an activation. No context, no residual — the minimal building block.

**Integrates into:** used as the innermost layer of `MLP` — every block in an `MLP` is a `LinearBlock`. Also used directly inside `ContinuousInputTokenizer` and `TokenwiseHead` as their final projection step.

**Use standalone when:** you need exactly one projection layer with a specific activation and no conditioning. If you need multiple layers, use `MLP` instead.

**Use for:** projection heads after encoders, dimension adapters between modules, single-layer score networks, embedding lookup projections.

In [16]:
block = LinearBlock(input_dim=64, output_dim=128, activation="silu")

# (batch, features) — LinearBlock broadcasts over any leading dims
# Real: tabular row (B, num_features)  e.g. (32, 64)
#       sequence token (B, T, features) e.g. (32, 16, 64) — broadcasts over T
#       latent code (B, latent_dim)      e.g. (8, 512)
x = torch.randn(4, 64)
out = block(x)
print(f"Input: {x.shape}  ->  Output: {out.shape}")   # (4, 128)
print(block)

Input: torch.Size([4, 64])  ->  Output: torch.Size([4, 128])
SiLU(Linear(x))


### 1.2 Context injection: `concat`

Concatenates `[x, context]` before the linear layer. The projection sees both jointly.

**Integrates into:** `MLP(context_injection="concat")` — the same concatenation happens at every layer. Used in conditioning adapters that need maximum expressive power over the context.

**Use standalone when:** you need a single conditioned projection where context and input features are complementary and you want the network to learn cross-feature interactions directly.

**Use for:** class-conditional projection heads, CLIP-to-task-space adapters, any setting where context is categorical or dense and you want full interaction modeling.

> Note: combining `concat` with `do_residual=True` requires `output_dim == input_dim + context_dim` since the internal `x` grows in width after concatenation.

In [17]:
block = LinearBlock(
    input_dim=64, output_dim=128,
    context_dim=32, context_injection="concat",
)

# (batch, features) input + (batch, context_dim) context
# Real context: class embedding (B, embed_dim)  e.g. (4, 32)
#               CLIP text vector (B, 512) projected down to context_dim
x   = torch.randn(4, 64)
ctx = torch.randn(4, 32)

out = block(x, ctx)   # internal: Linear([x, ctx]) -> (B, 128)
print(f"concat: x={x.shape}, ctx={ctx.shape} -> {out.shape}")

concat: x=torch.Size([4, 64]), ctx=torch.Size([4, 32]) -> torch.Size([4, 128])


### 1.3 Context injection: `add`

Projects context to `input_dim` and adds it element-wise to the input before the linear layer.

**Integrates into:** `MLP(context_injection="add")` — the same addition happens at every layer. Lightweight: adds only one extra linear projection per layer.

**Use standalone when:** context is an additive offset in feature space — e.g. a learned positional bias or a per-condition translation. Lower capacity than `concat` but cheaper.

**Use for:** learned positional biases, per-condition feature translations, lightweight speaker or domain offsets.

In [18]:
block = LinearBlock(
    input_dim=64, output_dim=64,
    context_dim=32, context_injection="add",
)

# (batch, features) — output_dim must equal input_dim when using add (additive offset)
# Real context: speaker ID embedding (B, 32), domain code (B, 32)
x   = torch.randn(4, 64)
ctx = torch.randn(4, 32)   # projected to input_dim=64 before adding

out = block(x, ctx)   # internal: Linear(x + proj(ctx)) -> (B, 64)
print(block)
print(f"add: -> {out.shape}")

SiLU(Linear(x + proj(context)))
add: -> torch.Size([4, 64])


### 1.4 Context injection: `multiply`

Projects context to `input_dim` and multiplies it element-wise — soft per-feature gating.

**Integrates into:** `MLP(context_injection="multiply")` — soft gating applied at every layer.

**Use standalone when:** context should selectively suppress or amplify specific input features, but you don't need the full affine (scale+shift) power of FiLM. Cheaper than FiLM (one projection instead of two).

**Use for:** attention-like feature selection, style modulation, input masking guided by a conditioning signal.

In [19]:
block = LinearBlock(
    input_dim=64, output_dim=128,
    context_dim=32, context_injection="multiply",
)

# (batch, features) — context acts as per-feature gate on x
# Real context: style vector (B, 32), sensor quality mask (B, 32)
x   = torch.randn(4, 64)
ctx = torch.randn(4, 32)   # projected to input_dim=64 then multiplied

out = block(x, ctx)   # internal: Linear(x * proj(ctx)) -> (B, 128)
print(block)
print(f"multiply: -> {out.shape}")

SiLU(Linear(x * proj(context)))
multiply: -> torch.Size([4, 128])


### 1.5 Context injection: `film`

Feature-wise Linear Modulation: context produces per-feature scale **and** shift.
Formula: `x_mod = x * (scale + 1) + shift` — an affine transformation conditioned on context.

**Integrates into:** `MLP(context_injection="film")` — FiLM applied at every layer. Also used inside `ConditionedConvBlock` (per-channel) and `ConditionedUNet` (conditioning flows through FiLM blocks).

**Use standalone when:** context is a continuous scalar or embedding that should modulate the feature distribution — e.g. a diffusion timestep, an audio pitch value, or a temperature parameter.

**Use for:** diffusion timestep conditioning, continuous sensor embeddings, per-sample temperature/style scaling. FiLM is the recommended default for any continuous conditioning signal.

> FiLM is the recommended default for continuous conditioning (timestep, sensor readings). Use `concat` if context is categorical and you want cross-feature interactions.

| Injection mode | Cost | Best for |
|---|---|---|
| `add` | 1 projection | additive offsets, positional biases |
| `multiply` | 1 projection | soft gating, feature selection |
| `film` | 2 projections (scale + shift) | continuous conditioning — **default** |
| `concat` | wider Linear layer | categorical context, max expressivity |
| `cross_attn` | full attention | rich context sets (text tokens) |

In [20]:
block = LinearBlock(
    input_dim=64, output_dim=128,
    context_dim=32, context_injection="film",
)

# (batch, features) input; context produces (scale, shift) each of shape (B, 64)
# Real context: sinusoidal time embedding (B, 32) from a diffusion scheduler
#               pitch embedding (B, 32) for voice conversion
x   = torch.randn(4, 64)
ctx = torch.randn(4, 32)

out = block(x, ctx)   # internal: Linear(x * (scale+1) + shift) -> (B, 128)
print(block)
print(f"film: -> {out.shape}")

SiLU(Linear(x * (scale + 1) + shift))
film: -> torch.Size([4, 128])


### 1.6 Context injection: `cross_attn`

Multi-head cross-attention: input is the query, context is key/value.
The block learns *which parts* of the context to attend to.

**Integrates into:** `MLP(context_injection="cross_attn")` — cross-attention applied at every layer. This is the most computationally expensive injection mode; use it only when the context is a variable-length set.

**Use standalone when:** context is a rich, multi-component set (e.g. token embeddings from a language model) and you want the network to selectively integrate only the relevant parts.

**Use for:** conditioning on text token sequences where only some words are relevant, multi-modal fusion where one modality queries another, attention-based context readout from a memory bank.

> `cross_attn` at the `LinearBlock` level is flat (non-hierarchical). For proper encoder-decoder cross-attention use `TransformerBlock(use_cross_attention=True)` from the transformer module.

In [21]:

block = LinearBlock(
    input_dim=64, output_dim=128,
    context_dim=64, context_injection="cross_attn",
)

# x is query (B, input_dim); ctx is key/value (B, context_dim)
# Real context: BERT token embeddings averaged to (B, 768) then projected to (B, 64)
#               memory bank embeddings (B, D)
x   = torch.randn(4, 64)
ctx = torch.randn(4, 64)   # projected to input_dim before attending

out = block(x, ctx)   # internal: Linear(CrossAttention(x, ctx)) -> (B, 128)
print(block)
print(f"cross_attn: -> {out.shape}")

SiLU(Linear(CrossAttention(x, context)))
cross_attn: -> torch.Size([4, 128])


### 1.7 Residual connections

When `do_residual=True`: `out = activation(Linear(conditioned_x) + x)`.
Requires `input_dim == output_dim` — the residual path bypasses the linear layer entirely.

**Integrates into:** `MLP(do_residual=True)` — residuals are automatically applied at layers where dimensions match; transition layers (dimension changes) never get residuals.

**Use standalone when:** the block should learn a *correction* on top of the input rather than a full transformation. Useful for refinement steps or the middle layers of a deep stack.

**Use for:** deep stacks (4+ layers) where gradient flow matters, refinement networks where input already carries most information, any block-style architecture.

In [22]:
block = LinearBlock(
    input_dim=128, output_dim=128,   # must match for residual
    do_residual=True, activation="gelu",
)

# (batch, features) — input_dim == output_dim required
x   = torch.randn(4, 128)
out = block(x)   # out = gelu(Linear(x) + x)
print(f"residual: {x.shape} -> {out.shape}")   # (4, 128)

# Mismatched dims raises an error at construction time
try:
    bad = LinearBlock(input_dim=64, output_dim=128, do_residual=True)
except ValueError as e:
    print(f"Expected error: {e}")

residual: torch.Size([4, 128]) -> torch.Size([4, 128])
Expected error: Residual connection requires input_dim (64) to equal output_dim (128).


---
## 2. `MLP` — stacked LinearBlocks

Composes multiple `LinearBlock` layers into a deep fully-connected network. Context injection mode, activations, and residual settings distribute across all layers automatically — you configure once and every block in the stack receives it.

**Integrates into:** used as the projection MLP inside `ContinuousInputTokenizer`, `TokenwiseHead`, and `PooledHead` (through the shared `make_head_mlp` helper). Also used as the backbone of score networks in diffusion pipelines.

**Use standalone when:** you need a multi-layer network (score predictor, encoder, regression head) and don't want to manually wire context to every `LinearBlock`. Use `LinearBlock` directly only when you need exactly one conditioned projection.

**Use for:** diffusion score networks, regression heads, encoder/decoder MLP stacks, per-token transformer heads, conditioning adapters (CLIP → task space).

**How to pick depth and width:**

| Pattern | `hidden_layers` | `hidden_dim` | When |
|---|---|---|---|
| Uniform width | default (`None`) | target width | most common — projection heads, score nets |
| Hourglass (expand→contract) | `[W, W*2, W]` | narrow | bottleneck autoencoders |
| Funnel (expand) | `[W, W*2, W*4]` | wide | decoder / upsampler |
| Bottleneck (compress) | `[W//2, W//4, W//2]` | original | latent compression |

`hidden_dim` is the output dimension of the **final** layer.
`hidden_layers` (length = `num_layers - 1`) controls intermediate dimensions.

### 2.1 Uniform hidden dimensions

All intermediate and final layers share the same width — the most common configuration.

**Integrates into:** this is the default `MLP` shape used inside `ContinuousInputTokenizer` and `TokenwiseHead` when `hidden_layers` is not specified.

**Use standalone when:** you want a fixed-width stack without a bottleneck or expansion. Most projection heads and score networks use this form.

**Use for:** projection heads after encoders, small regression networks, simple score networks, per-token classification heads.

In [23]:
mlp = MLP(
    input_dim=64,
    hidden_dim=256,   # all intermediate + final layers have this width
    num_layers=4,
    activation="silu",
)

# (batch, features) — uniform width: 64 -> 256 -> 256 -> 256 -> 256
# Real: tabular features (B, 64) -> score prediction (B, 256)
#       latent code (B, 64) -> projection head output (B, 256)
x   = torch.randn(4, 64)
out = mlp(x)   # (B, 256)
print(f"Input: {x.shape}  ->  Output: {out.shape}")   # (4, 256)
print(mlp)

Input: torch.Size([4, 64])  ->  Output: torch.Size([4, 256])
MLP (input_dim=64 -> hidden_dim=256)
  ├── Context: dim=0, Type=None
  └── Blocks Stack:
        [0] └── SiLU(Linear(x))
        [1] └── SiLU(Linear(x))
        [2] └── SiLU(Linear(x))
        [3] └── SiLU(Linear(x))


### 2.2 Custom hidden dimensions

`hidden_layers` controls the intermediate widths (length = `num_layers - 1`).

**Integrates into:** `MLP` with non-default `hidden_layers`. This is passed through directly to the `LinearBlock` stack.

**Use standalone when:** you need a specific width-to-depth profile — encoder bottleneck, decoder expansion, or hourglass shape.

**Use for:** bottleneck autoencoders (compress then reconstruct), VAE encoders/decoders, hierarchical feature extractors, capacity-controlled score networks.

| Shape | `hidden_layers` example | Typical use |
|---|---|---|
| Hourglass | `[256, 512, 256]` | bottleneck autoencoder |
| Funnel up | `[128, 256, 512]` | decoder / upsampler |
| Funnel down | `[512, 256, 128]` | encoder / compressor |

In [24]:
# Expanding then contracting (hourglass)
# Architecture: 64 -> 256 -> 512 -> 256 -> 128
mlp = MLP(
    input_dim=64, hidden_dim=128, num_layers=4,
    hidden_layers=[256, 512, 256],   # 3 intermediate dims for 4 layers
)
x   = torch.randn(4, 64)
out = mlp(x)   # (B, 128)
print(f"Hourglass MLP: {x.shape} -> {out.shape}")   # (4, 128)

# Bottleneck (compress then project)
# Architecture: 512 -> 128 -> 64 -> 128 -> 512
bottleneck = MLP(
    input_dim=512, hidden_dim=512, num_layers=4,
    hidden_layers=[128, 64, 128],
)
x2   = torch.randn(4, 512)
out2 = bottleneck(x2)   # (B, 512)
print(f"Bottleneck MLP: {x2.shape} -> {out2.shape}")   # (4, 512)

Hourglass MLP: torch.Size([4, 64]) -> torch.Size([4, 128])
Bottleneck MLP: torch.Size([4, 512]) -> torch.Size([4, 512])


### 2.3 Per-layer activations

`activation_list` assigns a different activation to each layer (length = `num_layers`).

**Integrates into:** `MLP` — the list is distributed one-to-one to each internal `LinearBlock`.

**Use standalone when:** the final layer must be linear (raw logits, unbounded regression targets) or you want a specific activation profile across the stack.

**Use for:** output layers for regression (`"identity"` on final layer), mixed-activation networks, any setting where you want to suppress squashing on the output.

In [25]:
mlp = MLP(
    input_dim=64, hidden_dim=64, num_layers=4,
    activation_list=["relu", "gelu", "silu", "identity"],   # final layer linear
)

# (batch, features) — "identity" on the last block means no output squashing
# Real: regression head predicting unbounded values (B, 64) -> (B, 64) logits
x   = torch.randn(4, 64)
out = mlp(x)   # (B, 64) — no final activation, values unbounded
print(f"Per-layer activations: {out.shape}")
print(f"Output range (no final squash): [{out.min().item():.2f}, {out.max().item():.2f}]")

Per-layer activations: torch.Size([4, 64])
Output range (no final squash): [-0.16, 0.17]


### 2.4 MLP with context conditioning

The context vector is injected at **every layer** — every `LinearBlock` inside the MLP receives the same context.

**Integrates into:** used as the conditioned score network in diffusion pipelines; the `context_injection` parameter passes straight through to each internal `LinearBlock`.

**Use standalone when:** you need a multi-layer conditioned network and don't want to wire context manually. All injection modes available on `LinearBlock` work identically here.

**Use for:** diffusion score networks (noisy latent + timestep → predicted noise), conditional regressors (features + sensor metadata → prediction), class-conditional MLP classifiers.

**How this relates to `LinearBlock`:** `MLP` simply wires `context_injection` to each of its internal `LinearBlock` layers — no extra mechanism.

| Mode | Recommended when |
|---|---|
| `film` | continuous context (timestep, scalar sensor) — default |
| `concat` | categorical context, maximum expressivity |
| `add` | lightweight additive offset at every layer |
| `cross_attn` | context is a set of tokens (expensive — avoid unless needed) |

In [26]:
# FiLM: most common for scalar/embedding context (continuous signals)
mlp = MLP(
    input_dim=64, hidden_dim=64, num_layers=4,
    context_dim=32, context_injection="film",
)
# (batch, features) input + (batch, context_dim) context injected at every layer
# Real: noisy latent (B, 64) + sinusoidal time embedding (B, 32) -> score (B, 64)
x   = torch.randn(4, 64)
ctx = torch.randn(4, 32)   # e.g. sinusoidal time embedding (B, 32)
out = mlp(x, ctx)   # (B, 64)
print(f"FiLM MLP: {out.shape}")

# concat: context expands each layer's input; most expressive but widens every Linear
mlp_concat = MLP(
    input_dim=64, hidden_dim=64, num_layers=3,
    context_dim=32, context_injection="concat",
)
out_concat = mlp_concat(x, ctx)   # (B, 64)
print(f"concat MLP: {out_concat.shape}")

FiLM MLP: torch.Size([4, 64])
concat MLP: torch.Size([4, 64])


### 2.5 Residual connections

When `do_residual=True`, each `LinearBlock` inside the MLP uses a residual wherever input and output dims match. Dimension transition layers never get residuals.

**Integrates into:** `MLP` — `do_residual=True` is forwarded to every internal `LinearBlock`; those where `in_dim != out_dim` silently skip the residual.

**Use standalone when:** building deep MLPs (6+ layers) where gradient flow is a concern, or when the network should learn corrections rather than full transformations.

**Use for:** deep score networks, residual refinement stacks, any multi-layer MLP where you want implicit skip connections.

In [27]:
# Uniform dims -> every layer gets a residual (all 6 blocks: 128->128)
mlp = MLP(
    input_dim=128, hidden_dim=128, num_layers=6, do_residual=True,
)
# (batch, features) — all layers same width, all get residuals
x   = torch.randn(4, 128)
out = mlp(x)   # (B, 128)
print(f"Uniform residual MLP: {out.shape}")

# Mixed dims: residuals only where dims align (256->256->256 segment)
# Architecture: 64 -> 256 -> 256 -> 256 -> 128  (only middle 3 layers get residuals)
mlp_mixed = MLP(
    input_dim=64, hidden_dim=128, num_layers=4,
    hidden_layers=[256, 256, 256], do_residual=True,
)
x2   = torch.randn(4, 64)
out2 = mlp_mixed(x2)   # (B, 128)
print(f"Mixed residual MLP: {out2.shape}")

Uniform residual MLP: torch.Size([4, 128])
Mixed residual MLP: torch.Size([4, 128])


### 2.6 MLP over sequences (broadcasts over token dim)

Both `LinearBlock` and `MLP` accept `(..., input_dim)` — they broadcast over any leading dimensions without reshaping.

**Integrates into:** this is exactly how `TokenwiseHead` works in the transformer module — it wraps an `MLP` and passes transformer output `(B, T, D)` directly without flattening.

**Use standalone when:** you want per-token or per-spatial-location processing without writing a loop or reshape. Both 3D `(B, T, D)` and 4D `(B, H, W, D)` tensors work out of the box.

**Use for:** per-token classification heads on transformer output, per-frame feature projection in video models, per-patch projection in ViT-style models, spatial feature map processing (channels-last).

In [28]:
mlp = MLP(input_dim=64, hidden_dim=32, num_layers=2)

# 2D: (batch, features) — standard tabular / latent input
print(f"2D: {torch.randn(4, 64).shape} -> {mlp(torch.randn(4, 64)).shape}")

# 3D: (batch, tokens, features) — each token projected independently
# Real: transformer output (B, T, 64) -> per-token logits (B, T, 32)
#       This is how TokenwiseHead works internally
print(f"3D: {torch.randn(4, 16, 64).shape} -> {mlp(torch.randn(4, 16, 64)).shape}")

# 4D: (batch, H, W, features) — spatial feature map, channels last
# Real: ViT patch features (B, H, W, 64) -> projected (B, H, W, 32)
print(f"4D: {torch.randn(4, 8, 8, 64).shape} -> {mlp(torch.randn(4, 8, 8, 64)).shape}")

2D: torch.Size([4, 64]) -> torch.Size([4, 32])
3D: torch.Size([4, 16, 64]) -> torch.Size([4, 16, 32])
4D: torch.Size([4, 8, 8, 64]) -> torch.Size([4, 8, 8, 32])
